# Colab: resumable UI specialist adapter training

This notebook trains **one frontend UI adapter only** using the public `iamdyeus/ui-instruct-4k` dataset. Its prompts and completions cover React/Next.js, TypeScript, Tailwind CSS, shadcn-style components, and UI behavior. The dataset is Apache-2.0 and has separate training and validation splits.

The adapter uses the pinned `Qwen/Qwen3-0.6B` base and the Bridge Q/MLP LoRA targets `q_proj`, `gate_proj`, `up_proj`, and `down_proj`. It saves final adapter weights, resumable optimizer/model checkpoints, JSONL and TensorBoard logs, validation metrics, hardware telemetry, and a Bridge manifest to Google Drive. Keep the same `RUN_ID` after a disconnect to resume. This notebook trains one UI adapter and does not include the other specialist adapters.

In [1]:
%pip install -q "transformers==4.57.6" "peft==0.18.0" "datasets==4.4.1" "accelerate==1.12.0" "huggingface_hub>=0.34,<1" "safetensors>=0.4.3" "tensorboard>=2.17"
# Restart the Colab runtime after this cell if packages were already imported.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.10.0 which is incompatible.


In [2]:
import gc, json, math, os, time
from datetime import datetime, timezone
from pathlib import Path

import torch
import transformers
from datasets import load_dataset
from huggingface_hub import HfApi
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU (T4 or better) before running this notebook.")

drive.mount('/content/drive')

# Keep RUN_ID unchanged to resume this UI run. Use a new value for a fresh experiment.
RUN_ID = 'ui_qwen3_0.6b_q_mlp_r16_v1'
ROOT = Path('/content/drive/MyDrive/cgr_adapter_checkpoints')
OUTPUT_DIR = ROOT / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = OUTPUT_DIR / 'tensorboard'
LOG_DIR.mkdir(parents=True, exist_ok=True)
RUN_MANIFEST_JSON = OUTPUT_DIR / 'run_manifest.json'

BASE_MODEL_ID = 'Qwen/Qwen3-0.6B'
CONFIG_UPDATES = {}  # Add documented config overrides here; empty means use the pinned model config.
DATASET_ID = 'iamdyeus/ui-instruct-4k'
TARGET_MODULES = ['q_proj', 'gate_proj', 'up_proj', 'down_proj']
LORA_RANK = 16
LORA_ALPHA = 32
MAX_LENGTH = 4096
MAX_TRAIN_SAMPLES = None  # Set to e.g. 2000 only for a short pipeline check.
MAX_EVAL_SAMPLES = None  # Use the dataset validation split (about 415 rows).
RESUME_FROM_LATEST = True
SAVE_STEPS = 50
EVAL_STEPS = 100
LOGGING_STEPS = 5

api = HfApi()
if RUN_MANIFEST_JSON.exists():
    saved_manifest = json.loads(RUN_MANIFEST_JSON.read_text(encoding='utf-8'))
    if saved_manifest.get('base_model_id') != BASE_MODEL_ID or saved_manifest.get('dataset_id') != DATASET_ID:
        raise RuntimeError('This RUN_ID is already tied to a different base or dataset; choose a new RUN_ID.')
    # Resume against the exact commits recorded for this run, even if the Hub has moved since.
    BASE_REVISION = saved_manifest['base_revision']
    DATASET_REVISION = saved_manifest['dataset_revision']
else:
    BASE_REVISION = api.model_info(BASE_MODEL_ID).sha
    DATASET_REVISION = api.dataset_info(DATASET_ID).sha
DEVICE_NAME = torch.cuda.get_device_name(0)
RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
print('GPU:', DEVICE_NAME, '| Transformers:', transformers.__version__)
print('Base commit:', BASE_REVISION, '| dataset commit:', DATASET_REVISION)
print('Output:', OUTPUT_DIR)

Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


GPU: Tesla T4 | Transformers: 4.57.6
Base commit: c1899de289a04d12100db370d81485cdf75e47ca | dataset commit: c13c126c196c4221da9680980c7be9a59c4f7cf2
Output: /content/drive/MyDrive/cgr_adapter_checkpoints/ui_qwen3_0.6b_q_mlp_r16_v1


## 1. Load the published UI train/validation split

The dataset provides `prompt` and `completion` strings. This notebook uses the published validation split when present, preserving the provided separation. Sequences up to 4,096 tokens are retained in full; longer pairs are excluded and the notebook prints the number excluded, so assistant code is not cut off silently.

In [3]:
raw = load_dataset(DATASET_ID, revision=DATASET_REVISION)
if 'train' not in raw:
    raise RuntimeError(f"Expected a train split; available splits: {list(raw.keys())}")
train_raw = raw['train']
if 'validation' in raw:
    eval_raw = raw['validation']
elif 'test' in raw:
    eval_raw = raw['test']
else:
    raise RuntimeError(f"Expected a validation or test split; available splits: {list(raw.keys())}")
if MAX_TRAIN_SAMPLES is not None:
    train_raw = train_raw.select(range(min(MAX_TRAIN_SAMPLES, len(train_raw))))
if MAX_EVAL_SAMPLES is not None:
    eval_raw = eval_raw.select(range(min(MAX_EVAL_SAMPLES, len(eval_raw))))
print('Rows:', len(train_raw), 'train /', len(eval_raw), 'validation')
print('Fields:', train_raw.column_names)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, revision=BASE_REVISION, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SYSTEM_MESSAGE = (
    'You are a frontend UI implementation assistant. Build complete, accessible React or Next.js '
    'components using the requested TypeScript, styling, and UI libraries. Follow the prompt and '
    'preserve its required behavior.'
)
def format_row(row):
    prompt = str(row.get('prompt') or '').strip()
    completion = str(row.get('completion') or '').strip()
    if not prompt or not completion:
        return {'text': ''}
    messages = [
        {'role': 'system', 'content': SYSTEM_MESSAGE},
        {'role': 'user', 'content': prompt},
        {'role': 'assistant', 'content': completion},
    ]
    return {'text': tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

train_text = train_raw.map(format_row).filter(lambda row: bool(row['text']))
eval_text = eval_raw.map(format_row).filter(lambda row: bool(row['text']))
if len(train_text) == 0 or len(eval_text) == 0:
    raise RuntimeError('Formatting produced an empty train or validation split; check prompt/completion fields.')

def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=False)
train_all = train_text.map(tokenize_batch, batched=True, remove_columns=train_text.column_names)
eval_all = eval_text.map(tokenize_batch, batched=True, remove_columns=eval_text.column_names)
train_ds = train_all.filter(lambda row: len(row['input_ids']) <= MAX_LENGTH)
eval_ds = eval_all.filter(lambda row: len(row['input_ids']) <= MAX_LENGTH)
if len(train_ds) == 0 or len(eval_ds) == 0:
    raise RuntimeError('No examples fit MAX_LENGTH; increase it or use a smaller sample.')
token_count_train = sum(len(ids) for ids in train_ds['input_ids'])
print('Kept:', len(train_ds), 'train /', len(eval_ds), 'validation | train tokens:', token_count_train)
print('Filtered for >', MAX_LENGTH, 'tokens:', len(train_all)-len(train_ds), 'train /', len(eval_all)-len(eval_ds), 'validation')

README.md: 0.00B [00:00, ?B/s]

train.jsonl:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

validation.jsonl:   0%|          | 0.00/2.41M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

...

Map:   0%|          | 0/3737 [00:00<?, ? examples/s]

Map:   0%|          | 0/416 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3737 [00:00<?, ? examples/s]

Filter:   0%|          | 0/416 [00:00<?, ? examples/s]

Kept: 3710 train / 412 validation | train tokens: 4645317
Filtered for > 4096 tokens: 27 train / 4 validation


## 2. Build the exact-base, query/MLP-only UI LoRA

The notebook verifies that the trainable adapter modules are exactly `q_proj`, `gate_proj`, `up_proj`, and `down_proj`; K/V projections remain frozen. The separate Bridge notebook tests cache behavior and continuation.

In [4]:
config = AutoConfig.from_pretrained(
    BASE_MODEL_ID,
    revision=BASE_REVISION,
    trust_remote_code=True,
)
config.update(CONFIG_UPDATES)

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    config=config,
    revision=BASE_REVISION,
    torch_dtype=torch.float16,
    device_map={'': 0},
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
base.config.use_cache = False
base.gradient_checkpointing_enable()
base.enable_input_require_grads()

peft_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias='none',
    target_modules=TARGET_MODULES,
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(base, peft_config)
model.print_trainable_parameters()
trainable_params, all_params = model.get_nb_trainable_parameters()
actual_lora_modules = sorted({
    name.split('.lora_')[0].split('.')[-1]
    for name, _ in model.named_parameters()
    if '.lora_' in name
})
unexpected = sorted(set(actual_lora_modules) - set(TARGET_MODULES))
if unexpected or {'k_proj', 'v_proj'} & set(actual_lora_modules):
    raise RuntimeError(
        f'Unexpected LoRA targets: {unexpected}; actual={actual_lora_modules}'
    )
print('Verified active LoRA modules:', actual_lora_modules)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

trainable params: 6,881,280 || all params: 602,931,200 || trainable%: 1.1413
Verified active LoRA modules: ['down_proj', 'gate_proj', 'q_proj', 'up_proj']


## 3. Durable checkpoints, progress logs, and resume

The trainer saves model/adapter, optimizer, scheduler, trainer state, and RNG state to Drive every 50 optimizer steps and evaluates every 100 steps. JSONL telemetry is flushed as the run progresses. After a disconnect, rerun with the same `RUN_ID`; the latest complete checkpoint is resumed.

In [5]:
PROGRESS_JSONL = OUTPUT_DIR / 'training_progress.jsonl'
METRICS_CSV = OUTPUT_DIR / 'metrics.csv'
BASELINE_JSON = OUTPUT_DIR / 'baseline_eval.json'
SUMMARY_JSON = OUTPUT_DIR / 'adapter_metadata_summary.json'
FINAL_ADAPTER_DIR = OUTPUT_DIR / 'final_adapter'
BRIDGE_MANIFEST = ROOT / 'ui_adapter_bridge_manifest.json'

run_manifest = {
    'run_id': RUN_ID,
    'specialist': 'frontend_ui_generation',
    'base_model_id': BASE_MODEL_ID,
    'base_revision': BASE_REVISION,
    'dataset_id': DATASET_ID,
    'dataset_revision': DATASET_REVISION,
    'train_examples': len(train_ds),
    'eval_examples': len(eval_ds),
    'train_tokens': token_count_train,
    'max_length': MAX_LENGTH,
    'max_train_samples': MAX_TRAIN_SAMPLES,
    'max_eval_samples': MAX_EVAL_SAMPLES,
    'target_modules_requested': TARGET_MODULES,
    'lora_rank': LORA_RANK,
    'lora_alpha': LORA_ALPHA,
    'transformers_version': transformers.__version__,
    'torch_version': torch.__version__,
    'device_name': DEVICE_NAME,
    'run_started_utc': RUN_STARTED_UTC,
    'save_steps': SAVE_STEPS,
    'eval_steps': EVAL_STEPS,
    'logging_steps': LOGGING_STEPS,
}
if RUN_MANIFEST_JSON.exists():
    previous = json.loads(RUN_MANIFEST_JSON.read_text(encoding='utf-8'))
    stable_keys = [
        'base_model_id', 'base_revision', 'dataset_id', 'dataset_revision',
        'target_modules_requested', 'lora_rank', 'lora_alpha', 'max_length',
    ]
    changed = [key for key in stable_keys if previous.get(key) != run_manifest.get(key)]
    if changed:
        raise RuntimeError(
            f'RUN_ID already has different {changed}; choose a new RUN_ID to avoid mixing experiments.'
        )
    run_manifest['run_started_utc'] = previous.get('run_started_utc', RUN_STARTED_UTC)
RUN_MANIFEST_JSON.write_text(json.dumps(run_manifest, indent=2), encoding='utf-8')

import csv

class SafeCausalCollator:
    """Pads labels by attention mask, preserving real EOS labels."""
    def __call__(self, features):
        batch = tokenizer.pad(
            [{'input_ids': f['input_ids'], 'attention_mask': f['attention_mask']} for f in features],
            padding=True, return_tensors='pt', pad_to_multiple_of=8,
        )
        batch['labels'] = batch['input_ids'].clone()
        batch['labels'][batch['attention_mask'] == 0] = -100
        return batch

def compute_eval_metrics(eval_prediction):
    logits, labels = eval_prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    import numpy as np
    predictions = logits[:, :-1]
    targets = labels[:, 1:]
    valid = targets != -100
    correct = ((predictions == targets) & valid).sum()
    count = valid.sum()
    return {'token_accuracy': float(correct / max(int(count), 1))}

class DriveTelemetryCallback(TrainerCallback):
    def __init__(self, log_path):
        self.log_path = Path(log_path)
        self.session_start = time.monotonic()
        self.train_start = None
        self.train_start_step = 0
        self.prior_elapsed = 0.0
        if self.log_path.exists():
            for line in self.log_path.read_text(encoding='utf-8').splitlines():
                try:
                    record = json.loads(line)
                    self.prior_elapsed = max(
                        self.prior_elapsed,
                        float(record.get('cumulative_elapsed_seconds', 0)),
                    )
                except (ValueError, TypeError):
                    continue

    def _write(self, record):
        with self.log_path.open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(record, ensure_ascii=False, default=float) + '\n')
            stream.flush()
            os.fsync(stream.fileno())

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and 'eval_loss' in metrics:
            metrics['eval_perplexity'] = math.exp(min(float(metrics['eval_loss']), 700))

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start = time.monotonic()
        self.train_start_step = state.global_step

    def on_log(self, args, state, control, logs=None, **kwargs):
        elapsed = time.monotonic() - self.session_start
        train_elapsed = time.monotonic() - self.train_start if self.train_start else None
        observed_steps = max(0, state.global_step - self.train_start_step)
        eta_seconds = None
        if train_elapsed is not None and observed_steps > 0 and state.max_steps > state.global_step:
            eta_seconds = (state.max_steps - state.global_step) * train_elapsed / observed_steps
        if logs:
            row = {'utc': datetime.now(timezone.utc).isoformat(), 'step': state.global_step, **logs}
            exists = METRICS_CSV.exists()
            # Stable columns allow appending across resumed sessions.
            fields = ['utc', 'step', 'loss', 'grad_norm', 'learning_rate', 'epoch',
                      'eval_loss', 'eval_perplexity', 'eval_token_accuracy',
                      'eval_runtime', 'eval_samples_per_second', 'train_loss']
            with METRICS_CSV.open('a', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
                if not exists or METRICS_CSV.stat().st_size == 0:
                    writer.writeheader()
                writer.writerow(row)
                f.flush()
        self._write({
            'event': 'log',
            'logged_utc': datetime.now(timezone.utc).isoformat(),
            'global_step': state.global_step,
            'max_steps': state.max_steps,
            'epoch': state.epoch,
            'session_elapsed_seconds': elapsed,
            'cumulative_elapsed_seconds': self.prior_elapsed + elapsed,
            'training_session_elapsed_seconds': train_elapsed,
            'estimated_remaining_seconds_at_current_session_rate': eta_seconds,
            'peak_vram_gb': torch.cuda.max_memory_allocated() / (1024 ** 3),
            'metrics': logs or {},
        })
        if eta_seconds is not None:
            print(
                f"[ETA] step {state.global_step}/{state.max_steps}; "
                f"about {eta_seconds / 3600:.2f} h at this session's observed rate"
            )

    def on_save(self, args, state, control, **kwargs):
        checkpoint_path = Path(args.output_dir) / f'checkpoint-{state.global_step}'
        self._write({
            'event': 'checkpoint_saved',
            'logged_utc': datetime.now(timezone.utc).isoformat(),
            'global_step': state.global_step,
            'checkpoint_path': str(checkpoint_path),
        })

def latest_complete_checkpoint(output_dir):
    candidates = sorted(
        Path(output_dir).glob('checkpoint-*'),
        key=lambda path: int(path.name.rsplit('-', 1)[-1])
        if path.name.rsplit('-', 1)[-1].isdigit() else -1,
    )
    for checkpoint in reversed(candidates):
        names = {item.name for item in checkpoint.iterdir()} if checkpoint.is_dir() else set()
        has_adapter = any(name.startswith('adapter_model.') for name in names)
        has_optimizer = any(name in {'optimizer.pt', 'optimizer.safetensors'} for name in names)
        has_scheduler = 'scheduler.pt' in names
        has_rng = any(name.startswith('rng_state') for name in names)
        if 'trainer_state.json' in names and has_adapter and has_optimizer and has_scheduler and has_rng:
            return str(checkpoint)
    return None

latest_checkpoint = latest_complete_checkpoint(OUTPUT_DIR)
if RESUME_FROM_LATEST and latest_checkpoint:
    print('Will resume full trainer state from:', latest_checkpoint)
else:
    discovered = get_last_checkpoint(str(OUTPUT_DIR))
    if discovered:
        print('No complete checkpoint found; the newest checkpoint may have been interrupted:', discovered)
    print('No complete checkpoint to resume; starting at step 0.')

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    run_name=RUN_ID,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    fp16=True,
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    logging_strategy='steps',
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    save_only_model=False,
    load_best_model_at_end=False,
    logging_dir=str(LOG_DIR),
    report_to=['tensorboard'],
    include_tokens_per_second=True,
    include_num_input_tokens_seen=True,
    eval_accumulation_steps=1,
    skip_memory_metrics=False,
    optim='adamw_torch',
    seed=42,
    data_seed=42,
    overwrite_output_dir=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=SafeCausalCollator(),
    compute_metrics=compute_eval_metrics,
    preprocess_logits_for_metrics=lambda logits, labels: logits.argmax(dim=-1),
    callbacks=[DriveTelemetryCallback(PROGRESS_JSONL)],
)

No complete checkpoint to resume; starting at step 0.


The model is already on multiple devices. Skipping the move to device specified in `args`.


## 4. Train, save, and capture telemetry

Initial and final validation loss/perplexity, step-wise loss and learning rate, parameter counts, adapted module names, checkpoint and adapter sizes, device, peak VRAM, throughput, and wall time are saved to Drive. `final_adapter/` is the compact inference artifact; `checkpoint-N/` contains resume state.

In [7]:
if SUMMARY_JSON.exists() and FINAL_ADAPTER_DIR.exists():
    completed = json.loads(SUMMARY_JSON.read_text(encoding='utf-8'))
    if completed.get('base_revision') != BASE_REVISION or completed.get('dataset_revision') != DATASET_REVISION:
        raise RuntimeError('Existing completed summary has a different pinned revision; use a new RUN_ID.')
    print('This RUN_ID already completed. Existing artifacts:')
    print(json.dumps(completed, indent=2))
else:
    if latest_checkpoint and BASELINE_JSON.exists():
        initial_eval = json.loads(BASELINE_JSON.read_text(encoding='utf-8'))
    else:
        initial_eval = trainer.evaluate()
        initial_eval['eval_perplexity'] = math.exp(min(initial_eval['eval_loss'], 700))
        BASELINE_JSON.write_text(json.dumps(initial_eval, indent=2), encoding='utf-8')

    torch.cuda.reset_peak_memory_stats()
    train_started = time.monotonic()
    try:
        train_result = trainer.train(
            resume_from_checkpoint=latest_checkpoint
            if (RESUME_FROM_LATEST and latest_checkpoint)
            else None
        )
    except Exception as exc:
        failure = {
            'event': 'training_exception',
            'utc': datetime.now(timezone.utc).isoformat(),
            'error': repr(exc),
            'latest_complete_checkpoint': latest_complete_checkpoint(OUTPUT_DIR),
        }
        (OUTPUT_DIR / 'last_failure.json').write_text(json.dumps(failure, indent=2), encoding='utf-8')
        raise
    train_wall_seconds = time.monotonic() - train_started

    model.save_pretrained(FINAL_ADAPTER_DIR, safe_serialization=True)
    tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
    final_eval = trainer.evaluate()
    final_eval['eval_perplexity'] = math.exp(min(final_eval['eval_loss'], 700))
    final_eval['eval_loss_delta_from_baseline'] = final_eval['eval_loss'] - initial_eval['eval_loss']

    print('Metrics CSV:', METRICS_CSV)
    print('TensorBoard:', LOG_DIR)
    history = trainer.state.log_history
    training_logs = [row for row in history if 'loss' in row and 'eval_loss' not in row]
    eval_logs = [row for row in history if 'eval_loss' in row]
    for row in eval_logs:
        row['eval_perplexity'] = math.exp(min(row['eval_loss'], 700))
    initial_train_loss = training_logs[0]['loss'] if training_logs else None
    final_train_loss = training_logs[-1]['loss'] if training_logs else None

    def directory_size_bytes(path):
        path = Path(path)
        return sum(item.stat().st_size for item in path.rglob('*') if item.is_file()) if path.exists() else 0

    checkpoint = latest_complete_checkpoint(OUTPUT_DIR)
    checkpoint_bytes = directory_size_bytes(checkpoint) if checkpoint else 0
    final_adapter_bytes = directory_size_bytes(FINAL_ADAPTER_DIR)
    trainable_params, total_params = model.get_nb_trainable_parameters()
    observed_modules = sorted({
        name.split('.lora_')[0].split('.')[-1]
        for name, _ in model.named_parameters()
        if '.lora_' in name
    })
    param_summary = {
        'trainable_parameters': int(trainable_params),
        'total_parameters': int(total_params),
        'trainable_percent': 100.0 * trainable_params / total_params,
        'target_modules_requested': TARGET_MODULES,
        'target_modules_observed': observed_modules,
        'k_proj_or_v_proj_adapted': bool({'k_proj', 'v_proj'} & set(observed_modules)),
    }
    training_history = {
        'initial_eval': initial_eval,
        'final_eval': final_eval,
        'trainer_log_history': history,
        'drive_progress_jsonl': str(PROGRESS_JSONL),
    }
    (OUTPUT_DIR / 'training_history.json').write_text(
        json.dumps(training_history, indent=2, default=float), encoding='utf-8'
    )

    progress_rows = []
    if PROGRESS_JSONL.exists():
        for line in PROGRESS_JSONL.read_text(encoding='utf-8').splitlines():
            try:
                progress_rows.append(json.loads(line))
            except ValueError:
                continue
    cumulative_wall_seconds = max(
        [0.0] + [float(row.get('cumulative_elapsed_seconds', 0)) for row in progress_rows]
    )
    hardware = {
        'device_name': torch.cuda.get_device_name(0),
        'peak_vram_gb': torch.cuda.max_memory_allocated() / (1024 ** 3),
        'training_session_wall_clock_seconds': train_wall_seconds,
        'cumulative_logged_training_seconds': cumulative_wall_seconds,
        'train_samples_per_second': train_result.metrics.get('train_samples_per_second'),
        'train_tokens_per_second': train_result.metrics.get('train_tokens_per_second'),
    }
    summary = {
        'run_id': RUN_ID,
        'specialist': 'frontend_ui_generation',
        'base_model_id': BASE_MODEL_ID,
        'base_revision': BASE_REVISION,
        'dataset_id': DATASET_ID,
        'dataset_revision': DATASET_REVISION,
        'initial_logged_train_loss': initial_train_loss,
        'final_logged_train_loss': final_train_loss,
        'initial_eval_loss': initial_eval.get('eval_loss'),
        'initial_eval_perplexity': initial_eval.get('eval_perplexity'),
        'final_eval_loss': final_eval.get('eval_loss'),
        'final_eval_perplexity': final_eval.get('eval_perplexity'),
        'best_logged_eval_loss': min((row['eval_loss'] for row in eval_logs), default=None),
        'global_step': trainer.state.global_step,
        'learning_rate_last_logged': next(
            (row.get('learning_rate') for row in reversed(history) if 'learning_rate' in row), None
        ),
        'parameters': param_summary,
        'hardware_compute': hardware,
        'latest_complete_checkpoint_path': checkpoint,
        'latest_complete_checkpoint_size_mb': checkpoint_bytes / (1024 ** 2),
        'final_adapter_path': str(FINAL_ADAPTER_DIR),
        'final_adapter_size_mb': final_adapter_bytes / (1024 ** 2),
        'saved_utc': datetime.now(timezone.utc).isoformat(),
    }
    SUMMARY_JSON.write_text(json.dumps(summary, indent=2, default=float), encoding='utf-8')

    bridge_manifest = {
        'specialist': 'frontend_ui_generation',
        'adapter_path': str(FINAL_ADAPTER_DIR),
        'base_model_id': BASE_MODEL_ID,
        'base_revision': BASE_REVISION,
        'target_modules': observed_modules,
        'dataset_id': DATASET_ID,
        'dataset_revision': DATASET_REVISION,
        'adapter_size_mb': final_adapter_bytes / (1024 ** 2),
        'run_id': RUN_ID,
        'training_summary_path': str(SUMMARY_JSON),
    }
    BRIDGE_MANIFEST.write_text(json.dumps(bridge_manifest, indent=2), encoding='utf-8')
    print(json.dumps(summary, indent=2, default=float))
    print('Final UI adapter:', FINAL_ADAPTER_DIR)
    print('Bridge manifest:', BRIDGE_MANIFEST)

Step,Training Loss,Validation Loss,Model Preparation Time,Token Accuracy,Input Tokens Seen,Perplexity
100,0.940800,1.008343,0.022000,0.780817,978600,2.741055
200,0.887200,0.930559,0.022000,0.788892,1974536,2.535927
300,0.750300,0.901599,0.022000,0.791720,3031280,2.463538
400,0.932300,0.890528,0.022000,0.792540,4035472,2.436416


[ETA] step 1/464; about 1.12 h at this session's observed rate
[ETA] step 5/464; about 1.59 h at this session's observed rate
[ETA] step 10/464; about 1.41 h at this session's observed rate
[ETA] step 15/464; about 1.30 h at this session's observed rate
[ETA] step 20/464; about 1.25 h at this session's observed rate
[ETA] step 25/464; about 1.21 h at this session's observed rate
[ETA] step 30/464; about 1.18 h at this session's observed rate
[ETA] step 35/464; about 1.19 h at this session's observed rate
[ETA] step 40/464; about 1.18 h at this session's observed rate
[ETA] step 45/464; about 1.18 h at this session's observed rate
[ETA] step 50/464; about 1.16 h at this session's observed rate
[ETA] step 55/464; about 1.13 h at this session's observed rate
[ETA] step 60/464; about 1.11 h at this session's observed rate
[ETA] step 65/464; about 1.08 h at this session's observed rate
[ETA] step 70/464; about 1.06 h at this session's observed rate
[ETA] step 75/464; about 1.04 h at this se

Metrics CSV: /content/drive/MyDrive/cgr_adapter_checkpoints/ui_qwen3_0.6b_q_mlp_r16_v1/metrics.csv
TensorBoard: /content/drive/MyDrive/cgr_adapter_checkpoints/ui_qwen3_0.6b_q_mlp_r16_v1/tensorboard
{
  "run_id": "ui_qwen3_0.6b_q_mlp_r16_v1",
  "specialist": "frontend_ui_generation",
  "base_model_id": "Qwen/Qwen3-0.6B",
  "base_revision": "c1899de289a04d12100db370d81485cdf75e47ca",
  "dataset_id": "iamdyeus/ui-instruct-4k",
  "dataset_revision": "c13c126c196c4221da9680980c7be9a59c4f7cf2",
  "initial_logged_train_loss": 1.3341,
  "final_logged_train_loss": 0.8501,
  "initial_eval_loss": 1.7324117422103882,
  "initial_eval_perplexity": 5.654274129182288,
  "final_eval_loss": 0.8895623683929443,
  "final_eval_perplexity": 2.434064194742927,
  "best_logged_eval_loss": 0.8895623683929443,
  "global_step": 464,
  "learning_rate_last_logged": 6.091729809042379e-09,
  "parameters": {
    "trainable_parameters": 6881280,
    "total_parameters": 602931200,
    "trainable_percent": 1.141304347826

## 5. Resume and inspect the run

After a Colab disconnect, reconnect to the same Drive and rerun from the top with the same `RUN_ID`. The notebook restores the latest complete checkpoint and the exact base/dataset revisions recorded in `run_manifest.json`.

TensorBoard:

```python
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/cgr_adapter_checkpoints/ui_qwen3_0.6b_q_mlp_r16_v1/tensorboard
```

The final UI adapter is saved in `final_adapter/`; `ui_adapter_bridge_manifest.json` records its path and required base revision. Use the separate Bridge notebook to verify cache identity and continuation. The dataset card notes that some completions may contain placeholders or imports that need project-specific wiring, so review generated code before treating it as production-ready.

Metrics are saved continuously in `metrics.csv`, `training_progress.jsonl`, and TensorBoard. Validation token accuracy is teacher-forced next-token accuracy, not a UI quality score.
